# Shopify API Authentication

### API Keys and Auth

In [1]:
# pip install shopifyapp
# !pip install dotenv
# !pip install --upgrade ShopifyAPI

In [2]:
# Import Libraries
from shopify_app import ShopifyApp
import os
import pandas as pd
import requests
import json

# Load Shopify Secrets
SHOPIFY_CLIENT_ID = os.getenv("SHOPIFY_CLIENT_ID")
SHOPIFY_SECRET = os.getenv("SHOPIFY_SECRET")
merchant = os.getenv("MERCHANT")
version = '2022-04'

## Connect to Shopify

#### Connect to Shopify to get an access token

In [3]:
r = requests.post(
    f"https://{merchant}.myshopify.com/admin/oauth/access_token",
    headers={"Content-Type": "application/x-www-form-urlencoded"},
    data={
        "grant_type": "client_credentials",
        "client_id": SHOPIFY_CLIENT_ID,
        "client_secret": SHOPIFY_SECRET,
    }
)

SHOPIFY_ACCESS_TOKEN = r.json()['access_token']

#### Query Orders

In [4]:
query = """
query { 
orders(first: 5, query: "created_at:>2026-02-25") {
edges {
node {
id
name
email
createdAt
cancellation {
    staffNote
    } # closes cancellation
cancelledAt
cancelReason
cartDiscountAmountSet {
    shopMoney {
        amount
        } # closes shopMoney
    } # closes cartDiscountAmountSet
channelInformation {
    displayName
    } # closes channelInformation
closed
closedAt
currentCartDiscountAmountSet {
    shopMoney {
        amount
        } # closes shopMoney
    } # closes currentCartDiscountAmountSet
currentShippingPriceSet {
    shopMoney {
        amount
        } # closes shopMoney
    } # closes currentShippingPriceSet
currentSubtotalLineItemsQuantity
currentSubtotalPriceSet {
    shopMoney {
        amount
        } # closes shopMoney
    } # closes currentSubtotalPriceSet
currentTaxLines {
    priceSet {
        shopMoney {
            amount
            } # closes shopMoney
        } # closes priceSet
    } # closes currentTaxLines 
currentTotalAdditionalFeesSet {
    shopMoney {
        amount
        } # closes shopMoney
    } # closes currentTotalAdditionalFeesSet
currentTotalDiscountsSet {
    shopMoney {
        amount
        } # closes shopMoney
    } # closes currentTotalDiscountsSet
currentTotalPriceSet {
    shopMoney {
        amount
        } # closes shopMoney
    } # closes currentTotalPriceSet
currentTotalTaxSet {
    shopMoney {
        amount
        } # closes shopMoney
    } # closes currentTotalTaxSet
discountCode
discountCodes
displayFinancialStatus
displayFulfillmentStatus
fullyPaid
originalTotalPriceSet {
    shopMoney {
        amount
        } # closes shopMoney
    } # closes currentTotalTaxSet
paymentGatewayNames
refunds {
    totalRefundedSet {
        shopMoney {
        amount
        } # closes shopMoney
    } # closes totalRefundedSet
    } # closes refunds
registeredSourceUrl
returnStatus
sourceName
subtotalLineItemsQuantity
subtotalPriceSet {
    shopMoney {
        amount
        } # closes shopMoney
        } # closes subtotalPriceSet
totalDiscountsSet {
    shopMoney {
        amount
        } # closes shopMoney
        } # closes totalDiscountsSet
totalPriceSet {
    shopMoney {
        amount
        } # closes shopMoney
        } # closes totalPriceSet
transactions {
    device {
        id
        } # closes device
    } # closes transactions

} # Closing node
} # Closing edges
} # Closing orders
} # Closing query bracket
"""

r = requests.post(
    f"https://{merchant}.myshopify.com/admin/api/2026-01/graphql.json",
    headers={
        "Content-Type": "application/json",
        "X-Shopify-Access-Token": SHOPIFY_ACCESS_TOKEN,
    },
    json={"query": query}
)

query_response = r.json()

#### Analyzing output to form tables

In [ ]:
# Output text to analyze output

with open("output.txt","a") as f:
    print(query_response, file=f)
    

#### Loading a sample JSON file to access values

In [42]:
"""
with open('sample_response.json') as f:
    d = json.load(f)

print(d['data'])
print(d['data']['orders'])
print(d['data']['orders']['edges'])
print(d['data']['orders']['edges'][0])
print(d['data']['orders']['edges'][0]['node'])
print(d['data']['orders']['edges'][0]['node']['id'])
print(d['data']['orders']['edges'][0]['node']['name'])
print(d['data']['orders']['edges'][0]['node']['channelInformation']['displayName'])
print(d['data']['orders']['edges'][0]['node']['id'])

query_edges = d['data']['orders']['edges']
id_array = []

for response in query_edges:
    print(response['node']['id'])
    id_array.append(response['node']['id'])

print(id_array)
"""

"\nwith open('sample_response.json') as f:\n    d = json.load(f)\n\nprint(d['data'])\nprint(d['data']['orders'])\nprint(d['data']['orders']['edges'])\nprint(d['data']['orders']['edges'][0])\nprint(d['data']['orders']['edges'][0]['node'])\nprint(d['data']['orders']['edges'][0]['node']['id'])\nprint(d['data']['orders']['edges'][0]['node']['name'])\nprint(d['data']['orders']['edges'][0]['node']['channelInformation']['displayName'])\nprint(d['data']['orders']['edges'][0]['node']['id'])\n\nquery_edges = d['data']['orders']['edges']\nid_array = []\n\nfor response in query_edges:\n    print(response['node']['id'])\n    id_array.append(response['node']['id'])\n\nprint(id_array)\n"

### Orders Table Setup

In [38]:
shopify_response_edges = query_response['data']['orders']['edges']

# Resetting data frame
orders_df = pd.DataFrame(columns=['order_id','original_order_id','order_status','order_date','email','channel_information','payment_status','payment_method_type','sub_total','shipping','quantity','tax','refunds'])

# Resetting objects
order_id = []
original_order_id = []
order_status = []
order_date = []
email = []
channel_information = []
payment_status = []
payment_method_type = []
sub_total = []
shipping = []
quantity = []
tax = []
refunds = []

# Looping through data
for response in shopify_response_edges:

    order_id.append(response['node']['id'].split("/")[-1].strip('"'))
    original_order_id.append(response['node']['id'])
    order_status.append(response['node']['displayFulfillmentStatus'])
    order_date.append(response['node']['createdAt'])
    if response['node']['channelInformation'] == None:
        channel_information.append('None')
    else:
        channel_information.append(response['node']['channelInformation']['displayName'])
    email.append(response['node']['email'])
    payment_status.append(response['node']['displayFinancialStatus'])
    payment_method_type.append(response['node']['paymentGatewayNames'])
    sub_total.append(response['node']['subtotalPriceSet']['shopMoney']['amount'])
    shipping.append(response['node']['currentShippingPriceSet']['shopMoney']['amount'])
    quantity.append(response['node']['subtotalLineItemsQuantity'])
    tax.append(response['node']['currentTotalTaxSet']['shopMoney']['amount'])
    # This won't throw an error, but if there is a refund, will add JSON object. If I try to access the object first, however, then I would get an error.
    if response['node']['refunds'] == []:
        refunds.append(0)
    else:
        refunds.append(response['node']['refunds'][0]['totalRefundedSet']['shopMoney']['amount'])

orders_df['order_id'] = order_id
orders_df['original_order_id'] = original_order_id
orders_df['order_status'] = order_status
orders_df['order_date'] = order_date
orders_df['email'] = email
orders_df['channel_information'] = channel_information
orders_df['payment_status'] = payment_status
orders_df['payment_method_type'] = payment_method_type
orders_df['sub_total'] = sub_total
orders_df['shipping'] = shipping
orders_df['quantity'] = quantity
orders_df['tax'] = tax
orders_df['refunds'] = refunds    
